<a href="https://colab.research.google.com/github/Sravani-939/genai-training-tasks/blob/main/video_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install
!apt update -y
!apt install ffmpeg -y
!pip install -q openai yt-dlp pydub faiss-cpu

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,863 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,862 kB]
Get:

In [3]:
import os
from google.colab import userdata
import math
import subprocess
import numpy as np
import faiss

from openai import OpenAI
from pydub import AudioSegment
from google.colab import files

# OpenAI key from Colab secrets
api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets")

client = OpenAI(api_key=api_key)

TRANSCRIBE_MODEL = "gpt-4o-mini-transcribe"
EMBED_MODEL = "text-embedding-3-small"
SUMMARY_MODEL = "gpt-4.1-mini"

os.makedirs("data", exist_ok=True)
os.makedirs("chunks", exist_ok=True)

# Clean old files so previous run does not get reused
for f in os.listdir("data"):
    if f.startswith("video.") or f == "audio.mp3" or f == "transcript.txt" or f == "summary_english.txt" or f == "faiss.index":
        os.remove(os.path.join("data", f))

for f in os.listdir("chunks"):
    if f.endswith(".mp3"):
        os.remove(os.path.join("chunks", f))

# 1. Get video: link or upload
choice = input("Type 'link' for video URL or 'upload' for file upload: ").strip().lower()

if choice == "link":
    url = input("Paste video URL: ").strip()

    subprocess.run([
        "yt-dlp",
        "-o", "data/video.%(ext)s",
        url
    ], check=True)

    video_candidates = [
        os.path.join("data", f)
        for f in os.listdir("data")
        if f.startswith("video.")
    ]

    if not video_candidates:
        raise ValueError("Video download failed")

    video_file = max(video_candidates, key=os.path.getmtime)

elif choice == "upload":
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded")

    name = list(uploaded.keys())[0]
    new_path = os.path.join("data", name)
    os.rename(name, new_path)
    video_file = new_path

else:
    raise ValueError("Invalid choice")

print("Video file used:", video_file)

# 2. Extract audio from video
audio_file = "data/audio.mp3"

subprocess.run([
    "ffmpeg", "-y", "-i", video_file,
    "-vn", "-ac", "1", "-ar", "16000", "-b:a", "64k",
    audio_file
], check=True)

print("Audio extracted:", audio_file)

# 3. Split audio if large
def split_audio(audio_path, max_mb=24):
    audio = AudioSegment.from_file(audio_path)
    size_mb = os.path.getsize(audio_path) / (1024 * 1024)

    if size_mb <= max_mb:
        return [audio_path]

    parts = math.ceil(size_mb / max_mb)
    part_len = len(audio) // parts
    out_files = []

    for i in range(parts):
        start = i * part_len
        end = len(audio) if i == parts - 1 else (i + 1) * part_len
        part = audio[start:end]
        out = f"chunks/part_{i}.mp3"
        part.export(out, format="mp3", bitrate="64k")
        out_files.append(out)

    return out_files

audio_parts = split_audio(audio_file)
print("Audio parts:", audio_parts)

# 4. Speech to text
full_text = ""

for part in audio_parts:
    with open(part, "rb") as f:
        transcript = client.audio.transcriptions.create(
            model=TRANSCRIBE_MODEL,
            file=f
        )
    full_text += transcript.text + "\n"

print(full_text[:2000])

with open("data/transcript.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

# 7. English summary from transcript
prompt = f"""
Below is a video transcript. The spoken language may be any language.
Create the final summary only in English.

Transcript:
{full_text}

Give:
1. Title
2. Short overview
3. Key points
4. Final short summary
"""

response = client.responses.create(
    model=SUMMARY_MODEL,
    input=prompt
)

summary = response.output_text
print(summary)

with open("data/summary_english.txt", "w", encoding="utf-8") as f:
    f.write(summary)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Type 'link' for video URL or 'upload' for file upload: link
Paste video URL: https://www.youtube.com/shorts/L4srCqJlVVY
Video file used: data/video.webm
Audio extracted: data/audio.mp3
Audio parts: ['data/audio.mp3']
Hey there, have you heard about generative AI? Let me break it down for you. Generative AI is a fascinating type of artificial intelligence that can actually create new content like text, images, and even music. So why do we call it generative? Well, it's because this AI doesn't just analyze existing data, but generates original stuff from scratch. That's pretty unique, right? What sets generative AI apart is its ability to mimic human creativity. It learns from tons of data and then produces content that feels like a human made it. This creative power makes it a game-changer in fields like writing, art, and music. It's pretty cool to see how this technology is evolving and transforming our world.

1. Title:  
Understanding Generative AI: The Future of Human-Like Creativit